In [1]:
import os

import numpy as np
import matplotlib.pyplot as plt
from bloqade.analog import load, save
from bloqade.analog.atom_arrangement import Square

if not os.path.isdir("data"):
    os.mkdir("data")

# setting the seed
rng = np.random.default_rng(1234)

durations = [0.3, 1.6, 0.3]

mis_udg_program = (
    Square(3, lattice_spacing=1.0)
    .apply_defect_density(0.3, rng=rng)
    .rydberg.rabi.amplitude.uniform.piecewise_linear(durations, [0.0, 1.0, 1.0, 0.0])
    .detuning.uniform.piecewise_linear(
        durations, [-30, -30, "final_detuning", "final_detuning"]
    )
)

# mis_udg_job = mis_udg_program.batch_assign(final_detuning=np.linspace(0, 80, 41))
mis_udg_job = mis_udg_program.batch_assign(final_detuning=np.linspace(0, 10, 5))


In [3]:
# from qbraid.runtime import QbraidProvider

In [5]:
# Modify job submission:

# provider = QbraidProvider()
# device = provider.get_device("quera_aquila_gpu")  # Hypothetical GPU backend
# hw_batch = device.run(mis_udg_job, shots=100, credits=5)  # Use 5 credits

In [ ]:
filename = os.path.join(os.path.abspath(""), "data", "MIS-UDG-job.json")

if not os.path.isfile(filename):
    # hw_batch = mis_udg_job.braket.local_emulator().run(shots=100)
    hw_batch = mis_udg_job.bloqade.python().run(shots=100)
    save(hw_batch, filename)

c:\Users\ryanp\anaconda3\envs\yqi\Lib\site-packages\scipy\integrate\_ode.py:438: UserWarning: dop853: problem is probably stiff (interrupted)
  self._y, self.t = mth(self.f, self.jac or (lambda: None),


RuntimeError: DOP853/DOPRI5: Problem is probably stiff (interrupted).

In [ ]:
batch = load(filename)

report = batch.report()

average_rydberg_excitation = report.rydberg_densities(filter_perfect_filling=False).sum(
    axis=1
)
final_detunings = report.list_param("final_detuning")

plt.plot(final_detunings, average_rydberg_excitation, color="#6437FF")
plt.xlabel("final detuning (rad/µs)")
plt.ylabel("total rydberg excitations")
plt.show()

plt.savefig("data/MIS-UDG.png")